# AlgoPay SDK — Zero-Config Demo (Run All & Watch the Dashboard)

This notebook demonstrates **every core feature** of the AlgoPay SDK with **zero manual setup**.
Just hit **Runtime → Run All** — credentials and dashboard connectivity are pre-configured.

**What you'll see:**
- Wallet creation on Algorand TestNet
- 6 guard types (budget, single-tx, rate limit, justification, recipient whitelist, confirm)
- Payment execution with guard enforcement
- Guard blocks with detailed reasons
- Payment intents (human-in-the-loop authorize/capture)
- Batch payments
- Ledger audit trail
- **Telemetry → Dashboard**: every event streams live to the hosted console

**After running, open the dashboard side-by-side:**
https://algopay-sdk-pay-b17a.vercel.app/dashboard/sdk-events

---

**Wallet details:** A TestNet wallet is auto-created (no funding needed for guard demos).
To run live on-chain transfers, fund the printed address at https://bank.testnet.algorand.network/

**Requirements:** `pip install algopay-sdk pandas matplotlib` (auto-installed in Cell 1)

In [ ]:
# Cell 0 — Install & Auto-Connect to Dashboard (zero-config)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "algopay-sdk", "pandas", "matplotlib"])

import os, json
from urllib.request import Request, build_opener, HTTPCookieProcessor
from urllib.error import HTTPError
from http.cookiejar import CookieJar

# ─── CONFIGURATION (edit these if using a different account) ───
DASHBOARD_URL = "https://algopay-sdk-pay-b17a.vercel.app"
DASHBOARD_EMAIL = "md.hatifosmani15@gmail.com"
DASHBOARD_PASSWORD = "Hatif@150503"
# ───────────────────────────────────────────────────────────────

jar = CookieJar()
opener = build_opener(HTTPCookieProcessor(jar))

def _api(method, path, body=None):
    url = f"{DASHBOARD_URL}{path}"
    data = json.dumps(body).encode() if body else None
    headers = {"Content-Type": "application/json"} if body else {}
    req = Request(url, data=data, headers=headers, method=method)
    try:
        with opener.open(req, timeout=30) as resp:
            raw = resp.read().decode()
            return resp.status, json.loads(raw) if raw else {}
    except HTTPError as e:
        raw = e.read().decode()
        return e.code, json.loads(raw) if raw else {"error": str(e)}

# 1. Login to dashboard
status, resp = _api("POST", "/api/auth/login", {"email": DASHBOARD_EMAIL, "password": DASHBOARD_PASSWORD})
assert status == 200, f"Dashboard login failed ({status}): {resp}"
print(f"✓ Logged into dashboard as {DASHBOARD_EMAIL}")

# 2. Get or create API key
_, keys_resp = _api("GET", "/api/api-keys")
existing_keys = keys_resp.get("keys", [])
if existing_keys:
    # We can't retrieve the full key after creation, so create a fresh one for this session
    print(f"  Found {len(existing_keys)} existing key(s), creating a fresh demo key...")

status, key_resp = _api("POST", "/api/api-keys", {"name": "Demo Notebook Key"})
API_KEY = key_resp.get("apiKey", "")
assert API_KEY.startswith("sk_"), f"API key creation failed: {key_resp}"
print(f"✓ API key created: {API_KEY[:16]}...")

# 3. Set environment for SDK telemetry
os.environ["ALGOPAY_CONSOLE_URL"] = DASHBOARD_URL
os.environ["ALGOPAY_API_KEY"] = API_KEY

print(f"\n✓ Dashboard connected — telemetry will stream to:")
print(f"  {DASHBOARD_URL}/dashboard/sdk-events")

In [ ]:
# Cell 1 — Populate Dashboard (policies, wallet, agent, gas pool, merchant, endpoint)
# This fills out the dashboard so every page has visible data during the demo.

# Spending policies
status, _ = _api("PATCH", "/api/workspace", {
    "network": "testnet",
    "maxDailyUsdc": "50.00",
    "maxSingleTxUsdc": "5.00",
    "approvalThresholdUsdc": "2.00",
    "requireJustification": True,
})
print(f"✓ Spending policies set (daily $50, single-tx $5, approval $2)")

# Wallet set + wallet on dashboard
_, sets_resp = _api("GET", "/api/wallet-sets")
wallet_sets = sets_resp.get("walletSets", [])
ws_id = next((s["id"] for s in wallet_sets if s.get("name") == "demo-agents"), None)
if not ws_id:
    _, ws_resp = _api("POST", "/api/wallet-sets", {"name": "demo-agents"})
    ws_id = ws_resp.get("id")
    print(f"✓ Wallet set 'demo-agents' created")
else:
    print(f"✓ Wallet set 'demo-agents' exists")

# Get or create a dashboard wallet
_, wallets_resp = _api("GET", "/api/wallets")
wallets_list = wallets_resp.get("wallets", [])
demo_wallets = [w for w in wallets_list if w.get("walletSetName") == "demo-agents"]
if demo_wallets:
    dash_wallet_address = demo_wallets[0]["address"]
    print(f"✓ Dashboard wallet exists: {dash_wallet_address[:12]}...")
else:
    _, w_resp = _api("POST", "/api/wallets", {"walletSetId": ws_id})
    dash_wallet_address = w_resp.get("address", "")
    print(f"✓ Dashboard wallet created: {dash_wallet_address[:12]}...")

# Gas pool
_, pools = _api("GET", "/api/gas-pools")
pool_list = pools if isinstance(pools, list) else []
if not pool_list:
    _, keys_r = _api("GET", "/api/api-keys")
    key_id = keys_r.get("keys", [{}])[0].get("id") if keys_r.get("keys") else None
    _, pool_resp = _api("POST", "/api/gas-pools", {
        "apiKeyId": key_id,
        "balanceUsdc": "10.00",
        "dailyCapCents": 5000,
        "alertThresholdUsdc": "2.00",
    })
    pool_id = pool_resp.get("id")
    print(f"✓ Gas pool created ($10 USDC)")
else:
    pool_id = pool_list[0]["id"]
    print(f"✓ Gas pool exists")

# Agent
_, agents = _api("GET", "/api/agents")
agent_list = agents if isinstance(agents, list) else []
if not agent_list and dash_wallet_address:
    _api("POST", "/api/agents", {
        "name": "Research Agent",
        "algoAddress": dash_wallet_address,
        "dailyLimitCents": 5000,
        "poolId": pool_id,
    })
    print(f"✓ Agent 'Research Agent' created")
else:
    print(f"✓ Agent(s) exist: {len(agent_list)}")

# Merchant
_, merchants = _api("GET", "/api/merchants")
merchant_list = merchants if isinstance(merchants, list) else []
if not merchant_list and dash_wallet_address:
    _api("POST", "/api/merchants", {"name": "Demo Merchant", "algoAddress": dash_wallet_address})
    print(f"✓ Merchant 'Demo Merchant' created")
else:
    print(f"✓ Merchant(s) exist: {len(merchant_list)}")

# Custom x402 endpoint
_, ep_resp = _api("GET", "/api/custom-endpoints")
endpoints = ep_resp.get("endpoints", [])
if not any(e.get("slug") == "demo-weather" for e in endpoints):
    _api("POST", "/api/custom-endpoints", {
        "slug": "demo-weather",
        "name": "Demo Weather API",
        "description": "x402 pay-per-call weather endpoint for agent demos",
        "endpointUrl": "https://x402.goplausible.xyz/examples/weather",
        "httpMethod": "GET",
    })
    print(f"✓ Custom endpoint 'demo-weather' created")
else:
    print(f"✓ Custom endpoint 'demo-weather' exists")

print(f"\n✓ Dashboard fully populated — open {DASHBOARD_URL}/dashboard to see all data")

In [ ]:
# Cell 2 — Initialize SDK & Register Pre-Funded Wallet
import base64
from algopay import AlgoPay
from algopay.core.types import Network, PaymentRequest, FeeLevel
import pandas as pd
from decimal import Decimal

# ─── FUNDED WALLET (persist across runs — fund this address once) ───
WALLET_PRIVATE_KEY_B64 = "DELsQ7/4HPEL/p3sLLFuEvqSW+6NceLhP4aLz9aH6/XquDnSLlJqmdQhaVQat/iQCcfRAB5NcI5roMvE4ZAynw=="
WALLET_ADDRESS = "5K4DTUROKJVJTVBBNFKBVN7YSAE4PUIADZGXBDTLUDF4JYMQGKP57O3P5Q"
WALLET_ID = "demo-funded-wallet"
WALLET_SET_ID = "demo-funded-set"
# ─── RECIPIENT: your dashboard wallet (receives real USDC) ───
RECIPIENT = "QDAQERDODYHDSSZGM275THOT3JRGZRQ6MSGZ2LOU5CV2P7JMFIBABBXRXM"
# ────────────────────────────────────────────────────────────────────

client = AlgoPay(network=Network.ALGORAND_TESTNET)
print(f"AlgoPay SDK initialized (network={client.config.network.value})")
print(f"Telemetry enabled: {client.telemetry.enabled}")
print(f"Console URL: {os.environ.get('ALGOPAY_CONSOLE_URL')}")
assert client.telemetry.enabled, "Telemetry not connected — check API key"
print()

# Register the pre-funded wallet
sk_bytes = base64.b64decode(WALLET_PRIVATE_KEY_B64)
client.wallet.repository.register_wallet(
    wallet_set_id=WALLET_SET_ID,
    wallet_id=WALLET_ID,
    address=WALLET_ADDRESS,
    private_key=sk_bytes,
    network_caip2=Network.ALGORAND_TESTNET.to_caip2(),
    name="demo-funded",
)

# USDC opt-in (idempotent — safe to re-run)
try:
    tx = client.wallet.opt_in_usdc(WALLET_ID)
    print(f"USDC opt-in TX: {tx}")
except Exception as e:
    print(f"USDC opt-in: already done or skipped ({e})")

# Use a simple namespace object so downstream cells can use wallet.id / wallet.address
class _W:
    def __init__(self, id, address, blockchain):
        self.id = id
        self.address = address
        self.blockchain = blockchain
wallet = _W(WALLET_ID, WALLET_ADDRESS, "algorand-testnet")

balance = await client.get_balance(wallet.id)
print(f"\nWallet Address: {wallet.address}")
print(f"USDC Balance:   {balance}")
print(f"Recipient:      {RECIPIENT}")
if balance == 0:
    print("\n⚠️  Wallet not funded yet! Fund it before running payment cells:")
    print(f"   1. ALGO (fees): https://bank.testnet.algorand.network/")
    print(f"   2. USDC (ASA 10458941): transfer from another testnet wallet")
    print(f"   Address: {WALLET_ADDRESS}")

In [ ]:
# Cell 3 — Configure Guards (Controllability)
# These guards protect the wallet before ANY payment can execute.

await client.add_budget_guard(wallet.id, daily_limit="5.00", name="daily_budget")
await client.add_single_tx_guard(wallet.id, max_amount="2.00", name="single_tx_cap")
await client.add_rate_limit_guard(wallet.id, max_per_hour=20, name="hourly_rate")
await client.add_justification_guard(wallet.id, min_length=3, name="justification")
await client.add_recipient_guard(
    wallet.id,
    mode="whitelist",
    addresses=[RECIPIENT],
    name="recipient_allowlist",
)

guards = await client.list_guards(wallet.id)
print("Active guards:")
for g in guards:
    print(f"  ✓ {g}")
print(f"\nTotal: {len(guards)} guards protecting wallet {wallet.id[:8]}...")
print(f"\nLimits: daily $5, single-tx $2, rate 20/hr")
print(f"Whitelisted recipient: {RECIPIENT[:12]}...")

In [ ]:
# Cell 4 — Happy Path Payment
# This payment passes ALL guards: within budget, under single-tx cap,
# has justification, and recipient is whitelisted.

result = await client.pay(
    wallet_id=wallet.id,
    recipient=RECIPIENT,
    amount="0.01",
    purpose="Weather API lookup for Tokyo forecast",
)

print(f"Success:    {result.success}")
print(f"Status:     {result.status.value}")
print(f"Amount:     {result.amount} USDC")
print(f"TX Hash:    {result.blockchain_tx}")
print(f"Guards OK:  {result.guards_passed}")
print(f"\n→ Dashboard received: payment.initiated → payment.guard_passed → payment.completed")

In [ ]:
# Cell 5 — Guard Block: Over Single-TX Limit
# The single-tx cap is $2.00 — this $3.00 payment will be BLOCKED.

over_budget = await client.pay(
    wallet_id=wallet.id,
    recipient=RECIPIENT,
    amount="3.00",
    purpose="Expensive market data feed",
)

print(f"Success:  {over_budget.success}")
print(f"Status:   {over_budget.status.value}")
print(f"Error:    {over_budget.error}")
print(f"\n→ Dashboard received: payment.initiated → payment.guard_blocked")
print(f"  Block reason visible on SDK Events page")

In [ ]:
# Cell 6 — Guard Block: Missing Justification
# JustificationGuard requires a non-empty purpose string.

no_purpose = await client.pay(
    wallet_id=wallet.id,
    recipient=RECIPIENT,
    amount="0.01",
    purpose="",  # Empty — blocked!
)

print(f"Success:  {no_purpose.success}")
print(f"Status:   {no_purpose.status.value}")
print(f"Error:    {no_purpose.error}")

In [ ]:
# Cell 7 — Guard Block: Recipient Not Whitelisted
# Only RECIPIENT is in the allowlist. This unknown address is blocked.

UNKNOWN_ADDR = "7ZUILL7ACGQ7G72RH3C2S5UJKB5IHXHI54ZRWLMG76Y347BGOZWBQBKIY"

wrong_recipient = await client.pay(
    wallet_id=wallet.id,
    recipient=UNKNOWN_ADDR,
    amount="0.01",
    purpose="Send to unknown address",
)

print(f"Success:  {wrong_recipient.success}")
print(f"Status:   {wrong_recipient.status.value}")
print(f"Error:    {wrong_recipient.error}")

In [ ]:
# Cell 8 — Guard Block: Rate Limit
# Max 5 per hour. We'll fire rapid payments to trigger the limit.

rate_results = []
for i in range(6):
    r = await client.pay(
        wallet_id=wallet.id,
        recipient=RECIPIENT,
        amount="0.01",
        purpose=f"Rate limit test #{i+1}",
    )
    rate_results.append({"attempt": i + 1, "success": r.success, "status": r.status.value})
    print(f"  Attempt {i+1}: {'✓' if r.success else '✗'} {r.status.value}")

print(f"\nRate limit triggered after {sum(1 for r in rate_results if r['success'])} successful payments")

In [ ]:
# Cell 9 — Payment Intents (Human-in-the-Loop)
# Create → Inspect → Confirm pattern (like Stripe's authorize/capture)

intent = await client.create_payment_intent(
    wallet_id=wallet.id,
    recipient=RECIPIENT,
    amount="0.02",
    purpose="Premium translation service",
)

print(f"Intent ID:     {intent.id}")
print(f"Status:        {intent.status.value}")
print(f"Amount:        {intent.amount} USDC")
print(f"Recipient:     {intent.recipient[:12]}...")
print("\n--- Human reviews and approves ---\n")

# Confirm the intent (capture)
capture = await client.confirm_payment_intent(intent.id)
print(f"Captured:      {capture.success}")
print(f"TX Hash:       {capture.blockchain_tx}")
print(f"Final Status:  {capture.status.value}")

In [ ]:
# Cell 10 — Batch Payments
# Execute 5 payments concurrently. Some will succeed, some will be blocked.

batch_requests = [
    PaymentRequest(wallet_id=wallet.id, recipient=RECIPIENT, amount=Decimal("0.01"), purpose="Batch: search API"),
    PaymentRequest(wallet_id=wallet.id, recipient=RECIPIENT, amount=Decimal("0.01"), purpose="Batch: weather API"),
    PaymentRequest(wallet_id=wallet.id, recipient=RECIPIENT, amount=Decimal("0.01"), purpose="Batch: translate"),
    PaymentRequest(wallet_id=wallet.id, recipient=RECIPIENT, amount=Decimal("3.00"), purpose="Batch: over single-tx limit"),
    PaymentRequest(wallet_id=wallet.id, recipient=RECIPIENT, amount=Decimal("0.01"), purpose=""),
]

batch_result = await client.batch_pay(batch_requests, concurrency=3)

print(f"Total:     {batch_result.total_count}")
print(f"Succeeded: {batch_result.success_count}")
print(f"Failed:    {batch_result.failed_count}")
print("\nResults:")
for i, r in enumerate(batch_result.results):
    status = r.status.value if r else "error"
    ok = r.success if r else False
    print(f"  [{i+1}] {'✓' if ok else '✗'} {status}")

In [ ]:
# Cell 11 — Ledger Query & Audit Trail (Observability)
# Every pay() call is recorded in the ledger — success, blocked, or failed.

entries = await client.ledger.query(wallet_id=wallet.id, limit=50)

rows = [
    {
        "status": e.status.value,
        "amount": str(e.amount),
        "purpose": (e.purpose or "")[:40],
        "recipient": (e.recipient or "")[:12] + "...",
        "tx_hash": (e.tx_hash or "")[:12],
        "timestamp": e.timestamp.strftime("%H:%M:%S"),
    }
    for e in entries
]

df = pd.DataFrame(rows)
print(f"Ledger entries: {len(entries)}")
print(f"Completed: {sum(1 for e in entries if e.status.value == 'completed')}")
print(f"Blocked:   {sum(1 for e in entries if e.status.value == 'blocked')}")
print(f"Failed:    {sum(1 for e in entries if e.status.value == 'failed')}")
print()
df

In [ ]:
# Cell 12 — Dashboard Live View
# All the events from this notebook are now visible on the dashboard.

events_sent = client.telemetry.events_sent

print("═" * 60)
print("  SWITCH TO DASHBOARD NOW")
print("  https://algopay-sdk-pay-b17a.vercel.app/dashboard/sdk-events")
print("═" * 60)
print(f"\nTelemetry events sent this session: {events_sent}")
print(f"\nEvery payment.initiated, guard_passed, guard_blocked,")
print(f"payment.completed, and payment.failed event from this")
print(f"notebook is now visible on the SDK Events page.")
print(f"\nYou will see:")
print(f"  • Green rows for completed payments")
print(f"  • Red rows for guard blocks (with reasons)")
print(f"  • Blue rows for initiated events")
print(f"  • Python language badges on every event")
print(f"  • Real-time auto-refresh every 5 seconds")
print(f"\nOther dashboard pages to explore:")
print(f"  • /dashboard         — Overview with KPIs")
print(f"  • /dashboard/wallets — Wallet sets & balances")
print(f"  • /dashboard/agents  — Agent management")
print(f"  • /dashboard/gas     — Gas pool status")
print(f"  • /dashboard/settings— API keys & spending policies")

In [ ]:
# Cell 13 — Summary Statistics
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

entries = await client.ledger.query(wallet_id=wallet.id, limit=100)

status_counts = {}
for e in entries:
    s = e.status.value
    status_counts[s] = status_counts.get(s, 0) + 1

total_spent = sum(e.amount for e in entries if e.status.value == "completed")

print(f"Total payments attempted: {len(entries)}")
print(f"Total USDC spent:        {total_spent}")
print(f"Success rate:            {status_counts.get('completed', 0)}/{len(entries)} ({100 * status_counts.get('completed', 0) / max(len(entries), 1):.0f}%)")
print(f"Guard blocks:            {status_counts.get('blocked', 0)}")
print(f"Telemetry events sent:   {client.telemetry.events_sent}")

# Pie chart
colors = {
    "completed": "#22c55e",
    "blocked": "#ef4444",
    "failed": "#f97316",
    "pending": "#3b82f6",
}
labels = list(status_counts.keys())
sizes = list(status_counts.values())
chart_colors = [colors.get(l, "#6b7280") for l in labels]

fig, ax = plt.subplots(figsize=(6, 4), facecolor="#0a0a0a")
ax.set_facecolor("#0a0a0a")
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=chart_colors, autopct="%1.0f%%",
    textprops={"color": "white", "fontsize": 11},
)
for t in autotexts:
    t.set_fontweight("bold")
ax.set_title("Payment Outcomes", color="white", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

import asyncio
await asyncio.sleep(3)

print(f"\nFinal telemetry count: {client.telemetry.events_sent} events sent to dashboard")
print("\n— Demo complete. Guards enforced policy, ledger captured everything, dashboard shows it all. —")